# 05 — Convert the Wikipedia-backed sources (Colab, CPU)

Converts **HoVer, FEVER, Natural Questions and FEVEROUS** to unified records (`data/public/<name>.jsonl` + `<name>.stats.json`). Their evidence lives in large Wikipedia dumps, which do not fit on a laptop disk; training never needs them, only the converted records.

- **Runtime:** CPU is enough (Runtime → Change runtime type → CPU). No GPU quota is used.
- **Disk:** ~75 GB at the peak (FEVEROUS alone: 10 GB archive + 53 GB database). The first code cell prints the free space.
- **Outputs** go to Google Drive (`MyDrive/jevlite/public/`), so a dead session loses only the dataset in progress.
- **Order:** HoVer → FEVER → NQ → FEVEROUS (the biggest last). Raw downloads stay in `/content/raw` for the session, so the full run after the smoke run does not download again.

Steps: run with `LIMIT = 200` (smoke), look at the samples, then set `LIMIT = None` and run the conversion cells again. Afterwards copy `MyDrive/jevlite/public/*` into `data/public/` on the laptop and rebuild the mixes there.

In [ ]:
# Environment: clone the code at a fixed commit, install without touching Colab's torch.
COMMIT = "main"   # set a commit hash for a reproducible run

!rm -rf /content/JEV_Judge && git clone -q https://github.com/Nefelim584/JEV_Judge.git /content/JEV_Judge
!cd /content/JEV_Judge && git checkout -q {COMMIT} && git log -1 --format='%h %ad %s' --date=short
!pip install -q --no-deps -e /content/JEV_Judge
!pip install -q loguru
!python --version
!df -h /content | tail -1

In [ ]:
# Parameters
LIMIT = 200                      # smoke run; None = full conversion
DATASETS = ["hover", "fever", "nq", "feverous"]
RAW = "/content/raw"            # dumps, kept for the session only

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/jevlite/public" if LIMIT is None else "/content/smoke"
!mkdir -p {OUT}
print("output:", OUT)

## Convert

One dataset per call, so a failure in one does not stop the others. Approximate full-run times: HoVer ~5 min, FEVER ~20 min (one pass over the 7.6 GB dump inside the zip), NQ ~30–40 min (37 parquet files), FEVEROUS ~40–60 min (download, unzip 53 GB, lookups).

If disk runs short before FEVEROUS, free the NQ files first: `!rm -rf /content/raw/nq`.

In [ ]:
import subprocess, sys
for name in DATASETS:
    cmd = [sys.executable, "/content/JEV_Judge/scripts/convert_public.py", "--datasets", name, "--out", OUT, "--raw-dir", RAW]
    if LIMIT is not None:
        cmd += ["--limit", str(LIMIT)]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=False, cwd="/content/JEV_Judge")
    !df -h /content | tail -1

## Check

Counts per split and label, then three random records per dataset. Things to look at: is the evidence text clean (no `-LRB-`, no `[[links]]`), does the state look like a retrieved chunk, are labels plausible.

In [ ]:
import json, random
from collections import Counter
from pathlib import Path

for name in DATASETS:
    path = Path(OUT) / f"{name}.jsonl"
    if not path.exists():
        print(f"== {name}: MISSING"); continue
    records = [json.loads(line) for line in path.open()]
    print(f"== {name}: {len(records)} records")
    print("  ", Counter((r["split"], r["type"], r["target"]) for r in records).most_common(12))
    for r in random.Random(0).sample(records, min(3, len(records))):
        state = r["state"] if isinstance(r["state"], str) else " ||| ".join(r["state"])
        print(f"  [{r['type']} → {r['target']}] {r['prompt']}")
        print("   ", state[:600].replace("\n", " / "))
    print()

## Hand-off

After the full run: download `MyDrive/jevlite/public/` (8 files) and put them into `data/public/` of the repo on the laptop. Then rebuild both mixes there:

```bash
uv run python scripts/build_mix.py --config configs/mixes/stage_a.yaml --out data/mix/stage_a --epochs 2
uv run python scripts/build_mix.py --config configs/mixes/stage_a_nosa.yaml --out data/mix/stage_a_nosa --epochs 2
```